# Lab 02 : 🧠 Les bases du machine learning : régression linéaire et descente de gradient

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atouammi/Formation_CNN_API_Docker_Module4/blob/main/Labs/02_ml_fundamentals.ipynb)

Bienvenue pour vos premiers pas dans le monde du machine learning ! Dans ce notebook, nous allons découvrir comment un ordinateur apprend à partir de données, en commençant par l'une des idées les plus puissantes et les plus intuitives de l'IA : **apprendre en s'améliorant**.

Nous allons décortiquer deux briques essentielles du machine learning moderne :

**🎯 Ce que vous allez apprendre :**
1. **La régression linéaire** – comment une machine trouve des tendances et fait des prédictions à l'aide de droites
2. **La descente de gradient** – comment un ordinateur améliore ses estimations par essais successifs
3. **Un exemple concret** – appliquer ces outils à une situation familière

Que vous soyez un débutant curieux ou que vous souhaitiez revoir les fondamentaux, ce notebook se veut :
- ✅ **Visuel** – avec des graphiques clairs pour montrer ce qui se passe
- ✅ **Interactif** – pour modifier les données et observer les résultats
- ✅ **Accessible** – pas besoin de maths avancées, juste de la curiosité

**📦 Au programme :**
- Une introduction en douceur aux idées clés
- Une progression pas à pas avec un minimum de prérequis
- Un mini-projet concret pour tout relier
- Du code à copier, enrichir et réutiliser

**💡 Pourquoi c'est important :**
Ces outils simples – la régression linéaire et la descente de gradient – sont la colonne vertébrale de nombreux systèmes d'IA. Les comprendre, c'est voir clairement comment un modèle apprend à partir des données, comment fonctionne l'optimisation et comment sont produites les prédictions.

> *De la prédiction du prix des maisons jusqu'au deep learning : c'est ici que tout commence.*

---

Prêt ? Apprenons aux machines à apprendre ! 🚀

In [1]:
# ⚙️ Configuration – exécutez cette cellule en premier.
# - Sur Google Colab : récupère les fichiers du cours (plotting_utils/, assets/, requirements.txt)
#   et installe ce qui manque.
# - En local (VS Code) : rien à faire, l'environnement est déjà installé avec requirements.txt.
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/Formation_CNN_API_Docker_Module4"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/atouammi/Formation_CNN_API_Docker_Module4.git {REPO_DIR}
    %cd {REPO_DIR}/Labs

if "." not in sys.path:
    sys.path.append(".")
print("Environnement :", "Google Colab" if IN_COLAB else "local")

Environnement : local


In [2]:
# Note pédagogique : cette cellule importe les bibliothèques et les fonctions utilitaires utilisées tout au long de la leçon.

# Mise en place rapide : importer nos outils. Exécutez cette cellule en premier (environ 10 secondes).

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Importer nos fonctions de visualisation
from plotting_utils.ml_fundamentals import (
    plot_house_data_scatter,
    create_manual_line_interactive,
    plot_computer_best_line,
    plot_learning_process,
    create_learning_rate_interactive,
    plot_coffee_productivity,
    normalize_data,
    denormalize_slope,
    denormalize_intercept
)

In [3]:
# Note pédagogique : fixer la graine aléatoire rend les données générées et l'entraînement reproductibles pendant la séance.

# Fixer la graine aléatoire pour des résultats reproductibles
np.random.seed(42)

## Partie 1 : La régression linéaire – trouver la tendance

### 🏠 Le défi du prix des maisons

Imaginez que vous êtes agent immobilier. Un client vous demande : *« À quel prix dois-je mettre en vente ma maison de 1 800 pieds carrés ? »* Vous disposez des données des ventes récentes. Comment trouver la tendance ?

La **régression linéaire** trouve la meilleure droite qui passe à travers les points – comme la « droite d'ajustement » que vous avez peut-être tracée à l'école, mais calculée automatiquement par l'ordinateur.


Commençons par créer des données réalistes de prix immobiliers. Elles nous aideront à visualiser le fonctionnement de la régression linéaire en pratique. Les surfaces sont en pieds carrés (sq ft, 1 sq ft ≈ 0,093 m²) et les prix en milliers de dollars.

In [4]:
# Note pédagogique : ce mini jeu de données immobilier donne une relation entrée-sortie concrète à modéliser.

house_sizes = np.array([800, 1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600])
house_prices = np.array([150, 180, 220, 250, 280, 320, 350, 380, 420, 450])

🧠 Que contiennent ces données ?

- Chaque maison est décrite par **une seule variable** (feature) : sa surface (en pieds carrés).
- La **cible** (target) que l'on cherche à prédire est le **prix** (en milliers de dollars).
- C'est un cas classique d'apprentissage supervisé : on veut apprendre une règle qui associe les entrées aux sorties.

Affichons les surfaces et les prix sous forme de paires pour voir avec quoi nous travaillons. Le premier élément est l'entrée (la surface), le second est la sortie (le prix).

In [5]:
# Note pédagogique : afficher les valeurs brutes aide à relier chaque point d'entraînement au nuage de points qui suit.

# Affichons les surfaces et les prix des maisons.
print("🏠 Données des ventes récentes :")
for size, price in zip(house_sizes, house_prices):
    print(f"   {size:,} sq ft → {price} k$")

🏠 Données des ventes récentes :
   800 sq ft → 150 k$
   1,000 sq ft → 180 k$
   1,200 sq ft → 220 k$
   1,400 sq ft → 250 k$
   1,600 sq ft → 280 k$
   1,800 sq ft → 320 k$
   2,000 sq ft → 350 k$
   2,200 sq ft → 380 k$
   2,400 sq ft → 420 k$
   2,600 sq ft → 450 k$


Les visualisations nous aident à mieux comprendre les données. Traçons les prix en fonction des surfaces : cela nous permettra peut-être de voir une tendance.

In [6]:
# Note pédagogique : le nuage de points montre visuellement la tendance avant d'ajuster le moindre modèle.

plot_house_data_scatter(house_sizes, house_prices)

🤔 Question : si vous deviez tracer une droite à travers ces points, où la placeriez-vous ?

### 🎮 Interactif : trouvez vous-même la meilleure droite !

À vous de jouer ! Voyons si vous arrivez à trouver la droite qui s'ajuste le mieux aux données. Le code suivant vous permet de régler manuellement la pente et l'ordonnée à l'origine d'une droite pour voir à quel point elle colle aux données.

- 🎯 Essayez différentes valeurs pour minimiser l'erreur !
- 💡 La « meilleure » droite est celle qui minimise l'erreur quadratique moyenne.

**Erreur quadratique** = l'écart entre les prix réels et les prix prédits par votre droite, élevé au carré pour éviter les valeurs négatives. L'objectif est de rendre cette erreur aussi petite que possible.

In [7]:
# Note pédagogique : la droite interactive permet de sentir comment la pente et l'ordonnée à l'origine modifient l'erreur de prédiction.

create_manual_line_interactive(house_sizes, house_prices)

interactive(children=(FloatSlider(value=0.0, description='Slope', max=1.0, step=0.01), FloatSlider(value=300.0…

### 🤖 Voyons maintenant comment l'ordinateur trouve la meilleure droite

Voyons comment l'ordinateur trouve automatiquement la meilleure droite grâce à la régression linéaire. Nous allons ajuster un modèle de régression linéaire simple aux données et visualiser le résultat. Il utilise la méthode de l'équation normale pour trouver la pente et l'ordonnée à l'origine optimales (une solution analytique au problème de régression linéaire).

La droite ressemble-t-elle à celle que vous avez tracée ? Sinon, pas d'inquiétude ! L'ordinateur suit une méthode systématique pour trouver le meilleur ajustement.

In [8]:
# Note pédagogique : scikit-learn calcule automatiquement la meilleure droite, ce qui donne une réponse de référence pour comparer.

# Entraîner le modèle de régression linéaire
model = LinearRegression()
X = house_sizes.reshape(-1, 1)  # Remise en forme pour sklearn
y = house_prices

model.fit(X, y)

# Récupérer les paramètres de la meilleure droite
best_slope = model.coef_[0]
best_intercept = model.intercept_
best_predictions = model.predict(X)
best_error = mean_squared_error(y, best_predictions)

# Visualiser le résultat
plot_computer_best_line(house_sizes, house_prices, best_slope, best_intercept, best_predictions, best_error)

L'ajustement a l'air bon ! Voyons comment interpréter cette droite :

In [9]:
# Note pédagogique : ces lignes traduisent les paramètres du modèle en langage courant.

print(f"   • Chaque pied carré supplémentaire ajoute {best_slope*1000:.0f} $ au prix")
print(f"   • Une maison de 0 pied carré coûterait {best_intercept*1000:.0f} $ (valeur de base)")
print(f"   • 🎯 Prix estimé d'une maison de 1 800 pieds carrés : {model.predict(np.array([[1800]]))[0]:.1f} k$")

   • Chaque pied carré supplémentaire ajoute 167 $ au prix
   • Une maison de 0 pied carré coûterait 15636 $ (valeur de base)
   • 🎯 Prix estimé d'une maison de 1 800 pieds carrés : 316.7 k$


Bravo ! Nous avons maintenant un modèle qui prédit le prix d'une maison à partir de sa surface. La meilleure droite trouvée par l'ordinateur nous donne une manière systématique d'estimer les prix, et nous voyons comment elle s'ajuste aux données. C'est ainsi que le machine learning nous aide à trouver des tendances dans les données et à faire des prédictions à partir de ces tendances. Si vous avez des questions ou envie d'aller plus loin, n'hésitez pas à demander ! 😊

Quelques points importants à garder en tête quand on laisse l'ordinateur trouver le modèle :
- **C'est nous** qui choisissons le type de modèle (ici, la régression linéaire). À nous de décider s'il est adapté à nos données.
- **C'est nous** qui définissons précisément ce que le modèle doit accomplir. Ici, on veut minimiser l'erreur quadratique entre les prix prédits et les prix réels. C'est ce qu'on appelle l'objectif d'entraînement.
- **C'est nous** qui définissons précisément comment le modèle atteint cet objectif. Ici, on trouve la solution en minimisant l'erreur quadratique grâce à la méthode de l'équation normale.
- **C'est nous** qui préparons les données utilisées pour entraîner le modèle. Ici, nous avons un petit jeu de données de surfaces et de prix ; en pratique, on voudrait un jeu de données plus grand et plus varié pour que le modèle généralise bien.

En résumé, c'est un exemple simple du fonctionnement du machine learning : on définit le modèle, l'objectif et les données, puis on laisse l'ordinateur trouver la meilleure solution. Un ordinateur n'est intelligent que dans la mesure où on le guide, et il a besoin de nous pour apprendre efficacement.

## Partie 2 : La descente de gradient – comment un ordinateur apprend

### 🏔️ L'analogie de la randonnée en montagne

L'ordinateur a trouvé la meilleure droite pour notre problème de prix immobiliers, mais **comment** ? Imaginez que vous randonnez dans un épais brouillard et que vous voulez atteindre le fond d'une vallée (l'erreur la plus faible). Vous ne voyez pas loin, mais vous sentez dans quelle direction le terrain descend. Alors vous :

1. **Tâtez le sol** autour de vos pieds (vous mesurez la pente)
2. **Faites un pas** vers le bas (vous ajustez votre position)
3. **Recommencez** jusqu'à atteindre le fond (vous trouvez l'erreur minimale)

C'est la **descente de gradient**, l'algorithme fondamental qui fait tourner la plupart des méthodes de machine learning !

**🎯 Pourquoi c'est important :** les réseaux de neurones apprennent grâce à la descente de gradient. Ils ajustent leurs paramètres (les poids) de façon itérative pour minimiser l'erreur entre les sorties prédites et les sorties réelles. C'est ainsi que les modèles de deep learning apprennent des structures complexes dans les données. Comprendre ce principe sur des exemples simples aide à comprendre le fonctionnement des modèles plus complexes.

### 🎬 La descente de gradient en action, pas à pas

Le code suivant simule la descente de gradient pour notre modèle de prix immobiliers. Il part d'une droite arbitraire et l'ajuste progressivement pour minimiser l'erreur.

Il faut faire un peu d'acrobatie avec les données pour que la descente de gradient fonctionne : l'algorithme n'est efficace que si les données sont normalisées (moyenne = 0, écart-type = 1). Cela l'aide à converger plus vite et de façon plus fiable. Il faut donc normaliser les données avant l'entraînement, puis « dénormaliser » les résultats après.

Voyons cela en action ! On lance la descente de gradient avec deux paramètres :
- **Le taux d'apprentissage** (learning rate) : la taille du pas que l'on fait vers le bas à chaque fois. Une petite valeur signifie de petits pas, une grande valeur de grands pas.
- **Le nombre d'itérations** : combien de fois on répète le processus « tâter le sol, faire un pas ».

In [10]:
# Note pédagogique : ce bloc implémente la descente de gradient pas à pas pour montrer comment la droite s'améliore au fil du temps.

# Normaliser les données pour un apprentissage stable
house_sizes_norm, mean_size, std_size = normalize_data(house_sizes)
mean_price = np.mean(house_prices)

def gradient_descent_demo(learning_rate, steps):
    """Démonstration pas à pas de la descente de gradient"""

    # Point de départ arbitraire
    slope = 0
    intercept = 0
    slope_orig = 0
    intercept_orig = 0

    # Suivi de la progression
    history = {'step': [], 'slope': [], 'intercept': [], 'error': []}

    print(f"🚀 Début de la descente de gradient (LR={learning_rate}, {steps} étapes)")
    print("Étape | Erreur  | Pente  | Ordonnée")
    print("-" * 35)

    for step in range(steps):
        # Calculer les prédictions et l'erreur
        predictions = slope * house_sizes_norm + intercept
        error = np.mean((house_prices - predictions) ** 2)

        # Revenir à l'échelle d'origine pour l'affichage
        slope_orig = denormalize_slope(slope, std_size)
        intercept_orig = denormalize_intercept(slope, mean_size, std_size, mean_price)

        # Enregistrer la progression
        history['step'].append(step)
        history['slope'].append(slope_orig)
        history['intercept'].append(intercept_orig)
        history['error'].append(error)

        # Afficher la progression toutes les 10 étapes
        if step % 10 == 0:
            print(f"{step:5d} | {error:7.2f} | {slope_orig:6.4f} | {intercept_orig:9.2f}")

        # Calculer les gradients (dans quelle direction aller)
        n = len(house_sizes_norm)
        errors = house_prices - predictions
        slope_gradient = -2 * np.sum(errors * house_sizes_norm) / n
        intercept_gradient = -2 * np.sum(errors) / n

        # Faire un pas vers le bas
        slope = slope - learning_rate * slope_gradient
        intercept = intercept - learning_rate * intercept_gradient

    print(f"\n🏁 Résultat final : pente={slope_orig:.4f}, ordonnée à l'origine={intercept_orig:.2f}")
    print(f"🎯 Comparaison avec la solution de l'ordinateur : pente={best_slope:.4f}, ordonnée à l'origine={best_intercept:.2f}")

    return history

# Lancer la démonstration
history = gradient_descent_demo(learning_rate=0.02, steps=50)

🚀 Début de la descente de gradient (LR=0.02, 50 étapes)
Étape | Erreur  | Pente  | Ordonnée
-----------------------------------
    0 | 99240.00 | 0.0000 |    300.00
   10 | 43867.97 | 0.0561 |    204.69
   20 | 19393.40 | 0.0933 |    141.33
   30 | 8575.58 | 0.1181 |     99.20
   40 | 3794.08 | 0.1346 |     71.19

🏁 Résultat final : pente=0.1446, ordonnée à l'origine=54.11
🎯 Comparaison avec la solution de l'ordinateur : pente=0.1673, ordonnée à l'origine=15.64


Vous voyez comment la pente et l'ordonnée à l'origine évoluent au fil des étapes, et comment l'erreur diminue à mesure que le modèle apprend. On constate aussi que l'on n'est pas encore tout à fait au fond de la vallée, mais qu'on s'en rapproche à chaque pas. Combien d'étapes supplémentaires faudrait-il, selon vous, pour l'atteindre ?

Une visualisation de la descente de gradient sera sans doute plus parlante.

### 📊 Visualiser le processus d'apprentissage

Vous allez « voir » l'apprentissage en action ! Le code suivant montre comment le modèle apprend au fil du temps : comment la pente et l'ordonnée à l'origine évoluent à chaque étape, et comment l'erreur diminue.

In [11]:
# Note pédagogique : ce graphique compare le chemin d'entraînement manuel à la solution de référence.

# Les lignes cibles sont ajoutées comme référence
plot_learning_process(history)

**🔍 Ce que vous voyez :**
- À gauche : l'erreur diminue à mesure que l'algorithme apprend (« on descend la pente »)
- À droite : les paramètres se rapprochent progressivement des valeurs optimales (lignes en pointillés). La pente semble déjà correcte dès le début, mais l'ordonnée à l'origine est encore loin de sa valeur optimale. Cela montre comment la descente de gradient améliore les paramètres du modèle de façon itérative.

C'est exactement ainsi que s'entraînent aussi les réseaux de neurones !

### 🎛️ Interactif : l'effet du taux d'apprentissage

Explorons l'influence du taux d'apprentissage sur la descente de gradient. Le taux d'apprentissage détermine la taille du pas que l'on fait vers le bas à chaque fois : petit taux, petits pas ; grand taux, grands pas.

Le code suivant vous permet de régler le taux d'apprentissage et d'observer son effet. Essayez différentes valeurs et regardez à quelle vitesse le modèle converge vers la solution optimale. Par défaut, le code effectue 50 itérations, mais vous pouvez modifier ce nombre pour voir comment le modèle apprend au fil du temps.

**🎯 Trouverez-vous un taux d'apprentissage qui fonctionne bien ?**

- Si le taux d'apprentissage est trop petit, le modèle met longtemps à converger.
- S'il est trop grand, le modèle peut dépasser la solution optimale et osciller autour, voire diverger.

In [12]:
# Note pédagogique : modifier le taux d'apprentissage de façon interactive montre pourquoi des pas trop petits ou trop grands posent problème.

create_learning_rate_interactive(house_sizes_norm, house_prices, std_size, mean_size, mean_price, denormalize_slope, denormalize_intercept)

interactive(children=(FloatLogSlider(value=0.01, description='Learning Rate', max=0.0, min=-4.0), IntSlider(va…

### 🌀 La descente de gradient, version stochastique !

La descente de gradient que nous venons de voir utilisait tous les points à la fois pour calculer la mise à jour de la pente et de l'ordonnée à l'origine. On parle de descente de gradient par lot complet (batch gradient descent).

Mais que se passerait-il avec des millions de points ? Le calcul des gradients prendrait énormément de temps à chaque étape !

La descente de gradient stochastique (SGD) accélère les choses en n'utilisant qu'un seul point à la fois pour mettre à jour les paramètres. Elle est plus rapide, consomme moins de mémoire, et généralise souvent mieux.

**🧠 Deux façons d'apprendre : descente de gradient par lot ou stochastique**

| Méthode                  | Utilise               | Avantages                            | Inconvénients                          |
|-------------------------|--------------------|----------------------------------|-------------------------------|
| **Descente de gradient par lot**     | Toutes les données à chaque étape | Pas stables, convergence nette | Peut être lente sur de grands jeux de données |
| **Descente de gradient stochastique** | Un point à la fois    | Mises à jour rapides, bonne généralisation | Trajectoire bruitée, moins stable       |


Le code suivant reprend le processus, mais cette fois avec la descente de gradient stochastique. Il met à jour la pente et l'ordonnée à l'origine à partir d'un seul point à la fois, ce qui accélère l'apprentissage. À votre avis, comment se compare-t-elle à la descente de gradient par lot vue précédemment ?

In [13]:
# Note pédagogique : cette démo de SGD fait une mise à jour par exemple, une alternative pratique à l'entraînement sur le lot complet.

def sgd_demo(learning_rate, steps):
    """Descente de gradient stochastique avec un point à la fois"""

    slope = 0
    intercept = 0
    slope_orig = 0
    intercept_orig = 0

    history = {'step': [], 'slope': [], 'intercept': [], 'error': []}

    print(f"🎯 Début de la SGD (LR={learning_rate}, {steps} étapes)")

    for step in range(steps):
        # Choisir un point au hasard
        idx = np.random.randint(0, len(house_sizes_norm))
        x_i = house_sizes_norm[idx]
        y_i = house_prices[idx]

        # Prédiction pour ce seul point
        prediction = slope * x_i + intercept
        error = (y_i - prediction) ** 2

        # Gradients pour ce seul point
        slope_grad = -2 * x_i * (y_i - prediction)
        intercept_grad = -2 * (y_i - prediction)

        # Mettre à jour les paramètres
        slope -= learning_rate * slope_grad
        intercept -= learning_rate * intercept_grad

        # Suivi à l'échelle d'origine
        slope_orig = denormalize_slope(slope, std_size)
        intercept_orig = denormalize_intercept(slope, mean_size, std_size, mean_price)

        history['step'].append(step)
        history['slope'].append(slope_orig)
        history['intercept'].append(intercept_orig)
        history['error'].append(error)

        if step % 10 == 0:
            print(f"{step:4d} | Erreur : {error:.2f} | Pente : {slope_orig:.4f} | Ordonnée : {intercept_orig:.2f}")

    print(f"\n🏁 Résultat final de la SGD : pente={slope_orig:.4f}, ordonnée à l'origine={intercept_orig:.2f}")
    return history

sgd_history = sgd_demo(learning_rate=0.03, steps=50)

🎯 Début de la SGD (LR=0.03, 50 étapes)
   0 | Erreur : 122500.00 | Pente : 0.0191 | Ordonnée : 267.55
  10 | Erreur : 18263.86 | Pente : 0.1054 | Ordonnée : 120.86
  20 | Erreur : 5599.41 | Pente : 0.1203 | Ordonnée : 95.47
  30 | Erreur : 3366.81 | Pente : 0.1572 | Ordonnée : 32.74
  40 | Erreur : 1141.08 | Pente : 0.1600 | Ordonnée : 27.92

🏁 Résultat final de la SGD : pente=0.1678, ordonnée à l'origine=14.82


Le résultat n'est pas tout à fait aussi bon qu'avec la descente de gradient par lot, mais n'oubliez pas que chaque étape ne s'appuie que sur un seul point. Avec 50 étapes, le modèle n'a donc vu que 50 points au total, alors que la descente par lot a utilisé tous les points 50 fois !

Visualisons à nouveau l'apprentissage pour voir comment le modèle apprend avec la descente de gradient stochastique. Le code suivant montre l'évolution de la pente, de l'ordonnée à l'origine et de l'erreur. Vous remarquerez que tout est un peu plus bruité qu'avec la descente par lot, mais le modèle apprend et s'améliore quand même au fil du temps.

In [14]:
# Note pédagogique : cette visualisation met en évidence la trajectoire plus bruitée, mais souvent plus rapide, de la descente de gradient stochastique.

plot_learning_process(sgd_history)


## Partie 3 : Une application concrète rapide (5 minutes)

### 📚 Prédire la productivité d'un data scientist

Appliquons la régression au quotidien d'un data scientist – et oui, cela inclut le café.

Dans cet exemple inspiré de la vie réelle, nous allons essayer de prédire combien de tâches un data scientist accomplit en fonction du nombre de tasses de café qu'il boit par jour. Une situation familière : un peu de caféine, un peu d'inspiration, peut-être un peu de chaos… mais existe-t-il une tendance mesurable ?

Notre jeu de données fictif suit la consommation quotidienne de café et le nombre de tâches réalisées. Vous allez :
- Examiner si la productivité augmente linéairement avec la consommation de café
- Entraîner un modèle linéaire simple sur une seule variable : le nombre de tasses de café
- Utiliser le modèle pour prédire la performance à de nouveaux niveaux de caféine (y compris des niveaux dangereusement élevés !)
- Visualiser la tendance et vous demander : plus, est-ce toujours mieux ?

C'est un exemple ludique, mais il reflète le type de modélisation exploratoire rapide qui lance de nombreux projets data réels. C'est l'occasion de mettre en pratique tout ce que vous avez appris dans un cadre familier et amusant – et qui sait, peut-être découvrirez-vous votre dose de café idéale en chemin. ☕

In [15]:
# Note pédagogique : cet exemple supplémentaire demande de transposer la même idée de régression à un nouveau jeu de données du quotidien.

# Créer des données fictives : consommation de café (tasses/jour) vs tâches réalisées
coffee_cups = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
tasks_done = np.array([2, 4, 6, 8, 10, 11, 11, 10, 9])  # Rendements décroissants !

print("☕ Données café et productivité :")
for cups, tasks in zip(coffee_cups, tasks_done):
    print(f"   {cups} tasse(s)/jour → {tasks} tâches réalisées")

# Entraîner le modèle de régression linéaire
coffee_model = LinearRegression()
coffee_model.fit(coffee_cups.reshape(-1, 1), tasks_done)

# Prédire la productivité pour une valeur donnée
cups_input = 6
predicted_tasks = coffee_model.predict(np.array([[cups_input]]))[0]

print(f"\n🎯 Prédiction : avec {cups_input} tasses de café par jour")
print(f"   → productivité prédite : {predicted_tasks:.1f} tâches/jour")

# Visualisation
plot_coffee_productivity(coffee_cups, tasks_done, coffee_model, cups_input, predicted_tasks)

☕ Données café et productivité :
   1 tasse(s)/jour → 2 tâches réalisées
   2 tasse(s)/jour → 4 tâches réalisées
   3 tasse(s)/jour → 6 tâches réalisées
   4 tasse(s)/jour → 8 tâches réalisées
   5 tasse(s)/jour → 10 tâches réalisées
   6 tasse(s)/jour → 11 tâches réalisées
   7 tasse(s)/jour → 11 tâches réalisées
   8 tasse(s)/jour → 10 tâches réalisées
   9 tasse(s)/jour → 9 tâches réalisées

🎯 Prédiction : avec 6 tasses de café par jour
   → productivité prédite : 8.9 tâches/jour


💡 À retenir :
- Chaque tasse de café supplémentaire ajoute 2 tâches réalisées… jusqu'à un certain point !
- Les données racontent une histoire non linéaire : la productivité plafonne (et peut même baisser !) au-delà d'environ 6 tasses.
- Un modèle linéaire est facile à ajuster, mais il ne capture ni les rendements décroissants ni, précisément, l'effet de départ.

👉 Un bon rappel : vérifiez toujours les hypothèses de votre modèle avant de faire confiance à ses prédictions !


## 🎉 Ce que vous avez accompli aujourd'hui !

En seulement 45 minutes, vous avez exploré les fondements du machine learning, les mêmes briques que celles des systèmes d'IA modernes utilisés partout, des moteurs de recommandation aux véhicules autonomes.

✅ **Concepts clés maîtrisés :**
1. **La régression linéaire** – utiliser des modèles simples pour révéler des tendances et faire des prédictions
2. **La descente de gradient** – comprendre comment une machine « apprend » en réduisant ses erreurs pas à pas
3. **Le cadrage concret** – appliquer des modèles à des problèmes familiers, et voir quand ils atteignent leurs limites

🚀 **Idées clés :**
- **Le machine learning commence simplement** – avec des droites et des gradients – mais ces outils passent à l'échelle jusqu'aux modèles les plus puissants.
- **Les hypothèses du modèle comptent** – un modèle linéaire est rapide et interprétable, mais il peut passer à côté de l'essentiel.
- **L'apprentissage est itératif** – les algorithmes s'améliorent pas à pas, comme nous.

🎯 **Prochaines étapes :**
1. **Essayez avec vos propres données** – appliquez ces concepts à des problèmes qui vous intéressent
2. **Découvrez d'autres algorithmes** – les réseaux de neurones dans le lab suivant (`03_neural_nets.ipynb`)


🌟 **Vue d'ensemble :**

Ce que vous avez vu aujourd'hui, et surtout la descente de gradient, n'est pas qu'un exercice d'école. C'est le cœur du deep learning, qui applique les mêmes principes pour entraîner des réseaux de neurones complexes sur d'immenses jeux de données.

Comprendre ces fondamentaux vous donne :
- La confiance nécessaire pour explorer des modèles plus complexes
- La capacité de déboguer et de démystifier ce qui se passe sous le capot
- Un regard lucide sur ce que le machine learning fait bien, et sur les cas où il faut rester prudent

👏 **Félicitations !** Vous comprenez maintenant les principes de base qui portent la révolution de l'IA. Les outils gagneront en complexité, mais les idées clés – tendances, apprentissage et optimisation – commencent ici.
